In [1]:
import numpy as np
import matplotlib.pyplot as plt

In [2]:
def get_transform_matrix(theta_degree: float, translation_matrix: np.array) -> np.array:
    theta_radians = np.radians(theta_degree)
    c, s = np.cos(theta_radians), np.sin(theta_radians)
    T = np.array([
        [c, -s, translation_matrix[0]],
        [s, c, translation_matrix[1]],
        [0, 0, 1],
    ])
    return T

In [3]:
def transform(point_body: np.array, theta_degree: float, translation_matrix: np.array) -> np.array:
    T = get_transform_matrix(theta_degree=theta_degree, translation_matrix=translation_matrix)
    point_body = np.append(point_body, 1)
    return T @ point_body

In [4]:
point_body = np.array([1, 1])
translate_matrix = np.array([1, 1])

T01 = transform(point_body, 45, translate_matrix)

In [5]:
point_3 = np.array([5,2,1])
relative_theta_degree = np.array([0, -30, 45])
relative_translation_matrix = np.array([
    [0, 0],
    [3, 3],
    [8, 5],
])
relative_T: list = []
final_T = np.eye(3)
for degree, m in zip(relative_theta_degree, relative_translation_matrix):
    t_relative = get_transform_matrix(degree, m)
    final_T = final_T @ t_relative
    relative_T.append(t_relative)
point_3_in_0 = final_T @ point_3
print(f'{point_3_in_0}')

[16.74019427  6.5560739   1.        ]


In [6]:
another_way = relative_T[2] @ point_3
another_way = relative_T[1] @ another_way
another_way = relative_T[0] @ another_way

another_way

array([16.74019427,  6.5560739 ,  1.        ])

In [7]:
from dataclasses import dataclass

@dataclass
class RotationAxes():
    X=1,
    Y=2,
    Z=3

def get3DTransfromMatrix(*, axes: RotationAxes, degree: float) -> np.array:
    theta_radians = np.radians(degree)
    c, s = np.cos(theta_radians), np.sin(theta_radians)
    
    if axes == RotationAxes.X:
        return np.array([
            [1, 0, 0],
            [0, c, -s],
            [0, s, c],
        ])
    if axes == RotationAxes.Y:
        return np.array([
            [c, 0, s],
            [0, 1, 0],
            [-s, 0, c],
        ])
    if axes == RotationAxes.Z:
        return np.array([
            [c, -s, 0],
            [s, c, 0],
            [0, 0, 1],
        ])

In [8]:
rX30 = get3DTransfromMatrix(axes=RotationAxes.X, degree=30)
rY45 = get3DTransfromMatrix(axes=RotationAxes.Y, degree=45)
rZ60 = get3DTransfromMatrix(axes=RotationAxes.Z, degree=60)

# first 30 in x -> second 45 in y -> last 60 in z
rZ60 @ rY45 @ rX30

array([[ 0.35355339, -0.5732233 ,  0.73919892],
       [ 0.61237244,  0.73919892,  0.28033009],
       [-0.70710678,  0.35355339,  0.61237244]])

In [9]:
def getHomogoneousTransformMatrix(axes: RotationAxes, degree: float, translate: np.array) -> np.array:
    translate = np.append(translate, 1).reshape(-1, 1)
    
    transfromMatrix = get3DTransfromMatrix(axes=axes, degree=degree)
    
    lastRow = np.zeros(3)
    transfromMatrix = np.vstack((transfromMatrix, lastRow))
    return np.hstack((transfromMatrix, translate))


In [10]:
rX30_t = getHomogoneousTransformMatrix(RotationAxes.X, 30, np.array([0, 0, 0]))
rY30_t = getHomogoneousTransformMatrix(RotationAxes.Y, 30, np.array([1, 2, -1]))
rY30_t @ rX30_t @ np.array([1, 2, 3, 1])

array([3.66506351, 2.23205081, 1.6160254 , 1.        ])

In [25]:
@dataclass
class JointData():
    def __init__(self, t: float, d_: float, a_: float, alpha_: float) -> None:
        self.theta = t
        self.d = d_
        self.a = a_
        self.alpha = alpha_

def getDHMatrix(jointData: JointData) -> np.array:
    theta_radians = np.radians(jointData.theta)
    alpha_radians = np.radians(jointData.alpha)
    a = jointData.a
    d = jointData.d
    
    c_theta, s_theta = np.cos(theta_radians), np.sin(theta_radians)
    c_alpha, s_alpha = np.cos(alpha_radians), np.sin(alpha_radians)
    
    return np.array([
        [c_theta, -s_theta, 0, a],
        [s_theta * c_alpha, c_theta * c_alpha, -s_alpha, -d * s_alpha],
        [s_theta * s_alpha, c_theta * s_alpha, c_alpha, d * c_alpha],
        [0, 0, 0, 1],
    ])

In [27]:
j1 = JointData(45, 0, 3, 0)
getDHMatrix(j1)

array([[ 0.70710678, -0.70710678,  0.        ,  3.        ],
       [ 0.70710678,  0.70710678, -0.        ,  0.        ],
       [ 0.        ,  0.        ,  1.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        ,  1.        ]])

In [29]:
def resolveDHMatrix(joinsData: np.array)  -> np.array:
    T = np.eye(4)
    for joint in joinsData:
        T = T @ getDHMatrix(joint)
    return T

joints = np.array([
    JointData(45, 0, 3, 0),
    JointData(45, 0, 2, 0),
    JointData(0, 0, 0.5, 0),
])
resolveDHMatrix(joints)

array([[ 4.26642159e-17, -1.00000000e+00,  0.00000000e+00,
         4.41421356e+00],
       [ 1.00000000e+00, -4.26642159e-17,  0.00000000e+00,
         1.91421356e+00],
       [ 0.00000000e+00,  0.00000000e+00,  1.00000000e+00,
         0.00000000e+00],
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         1.00000000e+00]])